# Concept-Aware Training — Google Colab Setup

**Before running:** Go to `Runtime > Change runtime type` and select **T4 GPU**.

### Quick overview of this notebook
1. Check GPU memory
2. Clone the repo and install dependencies
3. Mount Google Drive (for datasets)
4. Download the base model from Hugging Face
5. Run training — standard, synonym custom loss, or hypernym custom loss

> ⚠️ **T4 Memory Warning**: The T4 has ~15GB usable VRAM. Llama-3-8B in bfloat16 alone takes ~16GB — it will OOM. **Recommended**: use `meta-llama/Llama-3.2-1B` for T4, or `meta-llama/Meta-Llama-3-8B` only on Colab Pro (A100). A cell below lets you pick.

## 1. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

NVIDIA A100-SXM4-40GB, 40960 MiB, 40442 MiB

CUDA available: True
Device: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 2. Clone repo and install dependencies

Replace the repo URL below with your actual repo.

In [ ]:
# Clone the concept-aware training repo
REPO_URL = "https://github.com/SharvaGogawale1/concept-aware-training.git"

!git clone {REPO_URL} concept_aware_training
%cd concept_aware_training

Cloning into 'concept_aware_training'...
remote: Enumerating objects: 5006, done.
remote: Counting objects: 100% (5006/5006), done.
remote: Compressing objects: 100% (3636/3636), done.
remote: Total 5006 (delta 1324), reused 4980 (delta 1316), pack-reused 0 (from 0)
Receiving objects: 100% (5006/5006), 21.49 MiB | 11.69 MiB/s, done.
Resolving deltas: 100% (1324/1324), done.
/content/concept_aware_training


In [ ]:
# ── Fix 1: skip requirements.txt (it's a Conda lock file, not pip-compatible)
# ── Fix 2: use transformers>=4.41.0 to satisfy sentence-transformers

!pip install -q "transformers>=4.41.0,<5.0.0" datasets accelerate evaluate

# torch is already installed in Colab — no need to reinstall it
# Uncomment below ONLY if you get a torch import error:
# !pip install -q torch --index-url https://download.pytorch.org/whl/cu118

import transformers, torch
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 148.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
transformers: 4.57.6
torch: 2.10.0+cu128
CUDA: True


## 3. Mount Google Drive (for datasets)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_DATASET_ROOT = "/content/concept_aware_training/data/"

import os
print("Checking dataset directories...")
for subdir in ['syn', 'hyp']:
    path = os.path.join(DRIVE_DATASET_ROOT, subdir)
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"  ✅ {path} — {len(files)} files: {files}")
    else:
        print(f"  ❌ {path} — NOT FOUND. Check your Drive path.")

Checking dataset directories...
  ✅ /content/concept_aware_training/data/syn — 4 files: ['news', 'youtube', 'combined', 'arxiv']
  ✅ /content/concept_aware_training/data/hyp — 4 files: ['news', 'youtube', 'combined', 'arxiv']


In [ ]:
# Set train/val file paths — UPDATE these to match your actual filenames

# For synonym experiments
SYN_TRAIN = os.path.join(DRIVE_DATASET_ROOT, "syn/youtube/context_syn_train.txt")  # <-- UPDATE
SYN_VAL   = os.path.join(DRIVE_DATASET_ROOT, "syn/youtube/context_syn_val.txt")    # <-- UPDATE

# For hypernym experiments
HYP_TRAIN = os.path.join(DRIVE_DATASET_ROOT, "hyp/youtube/context_syn_train.txt")  # <-- UPDATE
HYP_VAL   = os.path.join(DRIVE_DATASET_ROOT, "hyp/youtube/context_syn_val.txt")    # <-- UPDATE

for label, path in [("SYN_TRAIN", SYN_TRAIN), ("SYN_VAL", SYN_VAL),
                    ("HYP_TRAIN", HYP_TRAIN), ("HYP_VAL", HYP_VAL)]:
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"{status}  {label}: {path}")

✅  SYN_TRAIN: /content/concept_aware_training/data/syn/youtube/context_syn_train.txt
✅  SYN_VAL: /content/concept_aware_training/data/syn/youtube/context_syn_val.txt
✅  HYP_TRAIN: /content/concept_aware_training/data/hyp/youtube/context_syn_train.txt
✅  HYP_VAL: /content/concept_aware_training/data/hyp/youtube/context_syn_val.txt


## 4. Download base model from Hugging Face

**T4 recommendation:** Use `meta-llama/Llama-3.2-1B` (~2.5 GB VRAM in bfloat16).  
**A100 / Colab Pro:** You can use `meta-llama/Meta-Llama-3-8B` (~16 GB).

Llama models require accepting the license on Hugging Face and logging in with a token.

In [ ]:
# Log in to Hugging Face (needed for gated models like Llama)
from huggingface_hub import login
login()  # Will prompt for your HF token — get it from https://huggingface.co/settings/tokens

In [ ]:
# Choose your model
# --------------------------------------------------
# T4 (Free Colab)   -> Llama-3.2-1B  (safe)
# T4 (tight)        -> Llama-3.2-3B  (may OOM)
# A100 (Pro)        -> Meta-Llama-3-8B
# --------------------------------------------------

MODEL_ID = "meta-llama/Llama-3.2-1B"  # <-- CHANGE if using 8B on Pro

# Download model to Colab local disk for fast training I/O
MODEL_LOCAL_PATH = f"/content/{MODEL_ID.split('/')[-1]}"

from huggingface_hub import snapshot_download
snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_LOCAL_PATH)
print(f"Model downloaded to: {MODEL_LOCAL_PATH}")

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

USE_POLICY.md: 0.00B [00:00, ?B/s]

LICENSE.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

original/tokenizer.model:   0%|          | 0.00/2.18M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Model downloaded to: /content/Llama-3.2-1B


In [ ]:
# Sanity check: confirm model files are present
!ls -lh {MODEL_LOCAL_PATH}

total 2.4G
-rw-r--r-- 1 root root  843 May  6 21:09 config.json
-rw-r--r-- 1 root root  185 May  6 21:09 generation_config.json
-rw-r--r-- 1 root root 7.6K May  6 21:09 LICENSE.txt
-rw-r--r-- 1 root root 2.4G May  6 21:09 model.safetensors
drwxr-xr-x 2 root root 4.0K May  6 21:09 original
-rw-r--r-- 1 root root  41K May  6 21:09 README.md
-rw-r--r-- 1 root root  301 May  6 21:09 special_tokens_map.json
-rw-r--r-- 1 root root  50K May  6 21:09 tokenizer_config.json
-rw-r--r-- 1 root root 8.7M May  6 21:09 tokenizer.json
-rw-r--r-- 1 root root 5.9K May  6 21:09 USE_POLICY.md


## 5. Set output directory

In [ ]:
# Save outputs to Drive so they persist after Colab disconnects
OUTPUT_ROOT = "/content/drive/MyDrive/concept_aware_outputs"  # <-- UPDATE if needed
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f"Outputs will be saved to: {OUTPUT_ROOT}")

Outputs will be saved to: /content/drive/MyDrive/concept_aware_outputs


## 6a. Run standard training (`run_clm.py`)

Use this to reproduce the baseline results.

In [ ]:
import subprocess

scripts = [
    "run_clm.py",
    "run_clm_syn_custom_loss.py",
    "run_clm_hyp_custom_loss.py"
]

script_dir = "/content/concept_aware_training/transformers/examples/pytorch/language-modeling"

for script in scripts:
    path = f"{script_dir}/{script}"
    try:
        with open(path, "r") as f:
            content = f.read()

        # Stub out the import
        content = content.replace(
            "from transformers.utils import check_min_version, send_example_telemetry",
            "from transformers.utils import check_min_version\ndef send_example_telemetry(*args, **kwargs): pass"
        )

        with open(path, "w") as f:
            f.write(content)

        print(f"✅ Patched: {script}")
    except FileNotFoundError:
        print(f"⚠️  Not found (skip): {script}")

✅ Patched: run_clm.py
✅ Patched: run_clm_syn_custom_loss.py
✅ Patched: run_clm_hyp_custom_loss.py


In [ ]:
!pip install -q bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00


In [ ]:
%cd /content/concept_aware_training/transformers/examples/pytorch/language-modeling/

# Standard CLM training (no custom loss)
# Using syn datasets here — swap for HYP_TRAIN/HYP_VAL if needed
!python3 run_clm.py \
  --model_name_or_path {MODEL_LOCAL_PATH} \
  --train_file {SYN_TRAIN} \
  --validation_file {SYN_VAL} \
  --save_total_limit 1 \
  --gradient_accumulation_steps 8 \
  --torch_dtype bfloat16 \
  --bf16 True \
  --block_size 128 \
  --per_device_train_batch_size 2 \
  --per_device_eval_batch_size 2 \
  --overwrite_output_dir \
  --do_train \
  --do_eval \
  --output_dir {OUTPUT_ROOT}/standard_syn


/content/concept_aware_training/transformers/examples/pytorch/language-modeling
2026-05-06 21:10:14.324447: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 21:10:14.394752: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO:__main__:Training/evaluation parameters TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafa

## 6b. Run synonym custom loss training (`run_clm_syn_custom_loss.py`)

1.   List item
2.   List item



In [ ]:
# Synonym custom loss — use the CSVs not the .txt
SYN_CUSTOM_TRAIN = os.path.join(DRIVE_DATASET_ROOT, "syn/youtube/context_loss_train.csv")
SYN_CUSTOM_VAL   = os.path.join(DRIVE_DATASET_ROOT, "syn/youtube/context_loss_val.csv")

# Hypernym custom loss
HYP_CUSTOM_TRAIN = os.path.join(DRIVE_DATASET_ROOT, "hyp/youtube/dict_loss_train.csv")
HYP_CUSTOM_VAL   = os.path.join(DRIVE_DATASET_ROOT, "hyp/youtube/dict_loss_val.csv")

import pandas as pd
df = pd.read_csv(SYN_CUSTOM_TRAIN)
print(df.columns.tolist())
print(df.head(2))


['text', 'context_syn']
                                        text  \
0                            Magicians ain’t   
1  Magicians ain’t shit unless they’re doing   

                                         context_syn  
0  ['nothing', 'worthless', 'useless', 'pathetic'...  
1  ['illusion', 'sorcery', 'enchantment', 'conjur...  


In [ ]:
from ast import literal_eval

bad_rows = []
for i, val in enumerate(df['context_syn']):
    try:
        literal_eval(val)
    except:
        bad_rows.append((i, val))

print(f"Bad rows: {len(bad_rows)}")
if bad_rows:
    print(bad_rows[:5])

Bad rows: 0


In [ ]:
trainer_path = "/content/concept_aware_training/transformers/examples/pytorch/language-modeling/custom_trainer.py"

with open(trainer_path, "r") as f:
    content = f.read()

# Patch the compute_loss signature to accept new kwargs
content = content.replace(
    "def compute_loss(self, model, inputs, return_outputs=False):",
    "def compute_loss(self, model, inputs, return_outputs=False, **kwargs):"
)

with open(trainer_path, "w") as f:
    f.write(content)

print("✅ Patched custom_trainer.py")

# Verify the patch
!grep -n "def compute_loss" {trainer_path}

✅ Patched custom_trainer.py
12:    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
92:#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
156:# #    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):


In [ ]:
trainer_path = "/content/concept_aware_training/transformers/examples/pytorch/language-modeling/custom_trainer.py"

with open(trainer_path, "r") as f:
    content = f.read()

# Replace deprecated self.tokenizer assignment with processing_class
content = content.replace(
    "self.tokenizer = tokenizer",
    "self.processing_class = tokenizer"
)

# Replace all internal uses of self.tokenizer with self.processing_class
content = content.replace(
    "self.tokenizer(",
    "self.processing_class("
)
content = content.replace(
    "self.tokenizer.bos_token_id",
    "self.processing_class.bos_token_id"
)
content = content.replace(
    "self.tokenizer.eos_token_id",
    "self.processing_class.eos_token_id"
)

with open(trainer_path, "w") as f:
    f.write(content)

print("✅ Patched custom_trainer.py")
!grep -n "tokenizer\|processing_class" {trainer_path}

✅ Patched custom_trainer.py
7:    def __init__(self, *args, completions_lookup=None, tokenizer=None, **kwargs):
10:        self.processing_class = tokenizer
26:            start_idx = (row == self.processing_class.bos_token_id).nonzero(as_tuple=True)[0]
27:            end_idx = (row == self.processing_class.eos_token_id).nonzero(as_tuple=True)[0]
45:                    token_ids = self.processing_class(word, return_tensors="pt", add_special_tokens=False)["input_ids"]
124:#                     token_ids = self.processing_class(word, return_tensors="pt", add_special_tokens=False)["input_ids"]
186:# #                tokens = [self.tokenizer.tokenize(word)[0] for word in t_star]
187:# #                completions = [self.tokenizer.convert_tokens_to_ids(token) for token in tokens]


In [ ]:
%cd /content/concept_aware_training/transformers/examples/pytorch/language-modeling/

!python3 run_clm_syn_custom_loss.py \
  --model_name_or_path {MODEL_LOCAL_PATH} \
  --train_file {SYN_CUSTOM_TRAIN} \
  --validation_file {SYN_CUSTOM_VAL} \
  --save_total_limit 1 \
  --gradient_accumulation_steps 8 \
  --torch_dtype bfloat16 \
  --bf16 True \
  --block_size 128 \
  --per_device_train_batch_size 2 \
  --per_device_eval_batch_size 2 \
  --overwrite_output_dir \
  --do_train \
  --do_eval \
  --output_dir {OUTPUT_ROOT}/syn_custom_loss

/content/concept_aware_training/transformers/examples/pytorch/language-modeling
2026-05-06 21:36:21.338396: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-06 21:36:21.409533: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO:__main__:Training/evaluation parameters TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafa

## 6c. Run hypernym custom loss training (`run_clm_hyp_custom_loss.py`)

In [ ]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="Trainer.tokenizer is now deprecated",
    category=FutureWarning
)

In [ ]:
%cd /content/concept_aware_training/transformers/examples/pytorch/language-modeling/

!python3 run_clm_hyp_custom_loss.py \
  --model_name_or_path {MODEL_LOCAL_PATH} \
  --train_file {HYP_CUSTOM_TRAIN} \
  --validation_file {HYP_CUSTOM_VAL} \
  --save_total_limit 1 \
  --gradient_accumulation_steps 8 \
  --torch_dtype bfloat16 \
  --bf16 True \
  --block_size 128 \
  --per_device_train_batch_size 2 \
  --per_device_eval_batch_size 2 \
  --overwrite_output_dir \
  --do_train \
  --do_eval \
  --output_dir {OUTPUT_ROOT}/hyp_custom_loss

Streaming output truncated to the last 5000 lines.
[WARNING|trainer.py:820] 2026-05-06 21:54:12,006 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:820] 2026-05-06 21:54:12,007 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:820] 2026-05-06 21:54:12,008 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:820] 2026-05-06 21:54:12,078 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:820] 2026-05-06 21:54:12,078 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:820] 2026-05-06 21:54:12,079 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:820] 2026-05-06 21:54:12,080 >> Trainer.tokenizer is now deprecated. You should use Trainer.processin

## 7. Check outputs

In [ ]:
# List saved checkpoints and trainer_state.json (contains train/eval loss)
for run in ['standard_syn', 'syn_custom_loss', 'hyp_custom_loss']:
    run_dir = os.path.join(OUTPUT_ROOT, run)
    if os.path.exists(run_dir):
        print(f"\n=== {run} ===")
        for f in os.listdir(run_dir):
            print(f"  {f}")
    else:
        print(f"\n[Not found yet] {run}")

In [ ]:
# Print final eval loss for each run
import json

for run in ['standard_syn', 'syn_custom_loss', 'hyp_custom_loss']:
    state_path = os.path.join(OUTPUT_ROOT, run, 'trainer_state.json')
    if os.path.exists(state_path):
        with open(state_path) as f:
            state = json.load(f)
        log_history = state.get('log_history', [])
        eval_logs = [x for x in log_history if 'eval_loss' in x]
        if eval_logs:
            last = eval_logs[-1]
            print(f"{run}: eval_loss={last['eval_loss']:.4f} (step {last['step']})")
    else:
        print(f"{run}: trainer_state.json not found")

---
## Troubleshooting

| Error | Fix |
|---|---|
| `CUDA out of memory` | Switch to `Llama-3.2-1B`, or reduce `--block_size` to 64 |
| `ModuleNotFoundError` | Run the pip install cell again, then restart runtime |
| `OSError: Can't load model` | Check `MODEL_LOCAL_PATH` — verify `ls` output |
| Drive datasets not found | Confirm path in `DRIVE_DATASET_ROOT` matches your Drive folder structure |
| Colab disconnects mid-run | Outputs go to Drive (`OUTPUT_ROOT`) so checkpoints are safe; re-run from the training cell |
| `requirements.txt` has Conda-only entries | Skip that cell; the explicit pip installs above cover core deps |

In [ ]:
!git config --global user.email "sharva.gogawale@gmail.com"
!git config --global user.name "SharvaGogawale1"


In [ ]:
%cd /content/concept_aware_training


/content/concept_aware_training


In [ ]:
!git status
!git diff --stat


On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   transformers/examples/pytorch/language-modeling/custom_trainer.py
	modified:   transformers/examples/pytorch/language-modeling/run_clm.py
	modified:   transformers/examples/pytorch/language-modeling/run_clm_hyp_custom_loss.py
	modified:   transformers/examples/pytorch/language-modeling/run_clm_syn_custom_loss.py
	new file:   transformers/examples/pytorch/language-modeling/wandb/latest-run
	new file:   transformers/examples/pytorch/language-modeling/wandb/run-20260506_204736-tnule0bu/files/config.yaml
	new file:   transformers/examples/pytorch/language-modeling/wandb/run-20260506_204736-tnule0bu/files/requirements.txt
	new file:   transformers/examples/pytorch/language-modeling/wandb/run-20260506_204736-tnule0bu/files/wandb-metadata.json
	new file:   transformers/examples/pytorch/language-modeling/wandb/run-20260506_204736-tnule0bu/files

In [ ]:
!git rm -r --cached transformers/examples/pytorch/language-modeling/wandb/

rm 'transformers/examples/pytorch/language-modeling/wandb/latest-run'
rm 'transformers/examples/pytorch/language-modeling/wandb/run-20260506_204736-tnule0bu/files/config.yaml'
rm 'transformers/examples/pytorch/language-modeling/wandb/run-20260506_204736-tnule0bu/files/requirements.txt'
rm 'transformers/examples/pytorch/language-modeling/wandb/run-20260506_204736-tnule0bu/files/wandb-metadata.json'
rm 'transformers/examples/pytorch/language-modeling/wandb/run-20260506_204736-tnule0bu/files/wandb-summary.json'
rm 'transformers/examples/pytorch/language-modeling/wandb/run-20260506_204736-tnule0bu/run-tnule0bu.wandb'
rm 'transformers/examples/pytorch/language-modeling/wandb/run-20260506_205401-y6lszdlf/files/config.yaml'
rm 'transformers/examples/pytorch/language-modeling/wandb/run-20260506_205401-y6lszdlf/files/requirements.txt'
rm 'transformers/examples/pytorch/language-modeling/wandb/run-20260506_205401-y6lszdlf/files/wandb-metadata.json'
rm 'transformers/examples/pytorch/language-model

In [ ]:
!echo "wandb/" >> .gitignore

In [ ]:
!git add .gitignore

In [ ]:
!git add transformers/examples/pytorch/language-modeling/custom_trainer.py
!git add transformers/examples/pytorch/language-modeling/run_clm.py
!git add transformers/examples/pytorch/language-modeling/run_clm_syn_custom_loss.py
!git add transformers/examples/pytorch/language-modeling/run_clm_hyp_custom_loss.py

In [ ]:
!git commit -m "Fix compatibility with newer transformers version"

[main f3c363b] Fix compatibility with newer transformers version
 5 files changed, 15 insertions(+), 11 deletions(-)
 create mode 100644 .gitignore


In [ ]:
from google.colab import userdata
REPO = "SharvaGogawale1/concept-aware-training"
!git remote set-url origin https://{GH_TOKEN}@github.com/{REPO}.git
!git push origin main

Enumerating objects: 19, done.
Counting objects: 100% (19/19), done.
Delta compression using up to 12 threads
Compressing objects: 100% (10/10), done.
Writing objects: 100% (11/11), 1.01 KiB | 1.01 MiB/s, done.
Total 11 (delta 9), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (9/9), completed with 8 local objects.
To https://github.com/SharvaGogawale1/concept-aware-training.git
   a955e75..f3c363b  main -> main
